# Pooling & Network Architecture

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/cnns/02-pooling-and-architectures

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — shrink, summarize, and stack

Convolutions detect features; **pooling** summarizes them. A pooling layer slides a window and keeps
one number per window — the **max** (strongest activation) or the **average** — which downsamples the
feature map, cuts computation, and adds a bit of **translation invariance** (a feature detected
slightly shifted still survives). Pooling and strided convs also grow the **receptive field**: how
much of the input each deep neuron "sees." A key architectural insight (VGG) is that **stacking small
3×3 convs** reaches the same receptive field as one big kernel but with fewer parameters and more
non-linearity. We build pooling from scratch and validate against `jax`.

## Max Pooling

In [ ]:
def max_pool(X, size=2, stride=2):
    H, W = X.shape
    OH = (H - size) // stride + 1
    OW = (W - size) // stride + 1
    out = np.zeros((OH, OW))
    for i in range(OH):
        for j in range(OW):
            out[i, j] = np.max(X[i*stride:i*stride+size, j*stride:j*stride+size])
    return out

feature_map = np.random.randn(8, 8)
pooled = max_pool(feature_map, 2, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(feature_map, cmap='magma')
axes[0].set_title(f'Feature Map ({feature_map.shape})', color='white')
axes[1].imshow(pooled, cmap='magma')
axes[1].set_title(f'After MaxPool ({pooled.shape})', color='white')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

**What to notice:** max pooling keeps the **strongest** activation in each window and throws away
the rest, halving the spatial size (2×2, stride 2). That's why it downsamples cheaply and gives small
**translation invariance** — nudging the input a pixel usually doesn't change which value is the max.

## The library way — validate pooling against `jax`

Frameworks implement pooling as a windowed reduction (`torch.nn.MaxPool2d`,
`jax.lax.reduce_window`). The cell runs `jax`'s windowed max and asserts it matches our from-scratch
`max_pool`.

In [ ]:
import jax, jax.numpy as jnp
from jax import lax

np.random.seed(0)
Xp = np.random.randn(8, 8)

def jax_maxpool(X, size=2, stride=2):
    return lax.reduce_window(jnp.asarray(X), -jnp.inf, lax.max,
                             (size, size), (stride, stride), 'VALID')

ours = max_pool(Xp, size=2, stride=2)
jx   = np.array(jax_maxpool(Xp))
print('our max_pool shape:', ours.shape, '| jax shape:', jx.shape)
assert np.allclose(ours, jx), "our max_pool must match jax.lax.reduce_window"
print('our max_pool == jax.lax.reduce_window (== torch.nn.MaxPool2d) ✓')

**What to notice:** identical output — our sliding-window max *is* what `nn.MaxPool2d` does under
the hood. The 8×8 input becomes 4×4, and every value is the max of its 2×2 block. Pooling is simple,
but getting the windowing exactly right (size, stride, padding) is where bugs hide.

## Receptive Field Growth

Stacking 3×3 convolutions increases the receptive field:

In [ ]:
layers = [1, 2, 3, 4, 5]
receptive = [3 + 2 * (l - 1) for l in layers]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(layers, receptive, color='#818cf8', alpha=0.8)
ax.set_xlabel('Number of 3×3 Conv Layers')
ax.set_ylabel('Receptive Field')
ax.set_title('Receptive Field Grows with Depth', color='white')
for l, r in zip(layers, receptive):
    ax.text(l, r + 0.3, f'{r}×{r}', ha='center', color='#94a3b8', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Why VGG uses only 3x3: two 3x3 == one 5x5 receptive field, with fewer params.
# Receptive field of L stacked f x f stride-1 convs:  RF = 1 + L*(f-1)
def rf(L, f): return 1 + L * (f - 1)
print('RF of two 3x3 =', rf(2, 3), ' vs one 5x5 =', rf(1, 5), ' (equal)')
print('RF of three 3x3 =', rf(3, 3), ' vs one 7x7 =', rf(1, 7), ' (equal)\n')

# Parameter count for C in- and out-channels: f*f*C*C per conv
def params(f, C, n=1): return n * f * f * C * C
for C in [32, 64]:
    one5 = params(5, C, 1)
    two3 = params(3, C, 2)
    print(f'C={C}: one 5x5 = {one5:,}   two 3x3 = {two3:,}   '
          f'saving = {one5 - two3:,} ({100*(one5-two3)/one5:.0f}% fewer)')
print('\n-> same receptive field, ~28% fewer parameters, plus an extra ReLU non-linearity.')


**What to notice:** the **receptive field** grows with depth — each layer sees a wider patch of the
original image. The VGG cell makes the key point: **two stacked 3×3 convs** cover the same 5×5
receptive field as one 5×5 conv, but with fewer parameters (`2·9 = 18` vs `25`) *and* an extra
non-linearity. That's why modern CNNs favor deep stacks of tiny kernels.

## Max vs average pooling

Pooling downsamples each window. **Max** keeps the strongest activation (edges/texture); **average** smooths. **Global average pooling** collapses each channel to one number.

In [ ]:
fmap = np.array([[1, 3, 2, 4],
                 [5, 6, 1, 2],
                 [7, 2, 3, 0],
                 [1, 2, 4, 8]], dtype=float)

def pool(x, k=2, mode='max'):
    H, W = x.shape
    out = np.zeros((H // k, W // k))
    for i in range(0, H, k):
        for j in range(0, W, k):
            win = x[i:i+k, j:j+k]
            out[i//k, j//k] = win.max() if mode == 'max' else win.mean()
    return out

print('max pool:\n', pool(fmap, 2, 'max'))
print('avg pool:\n', pool(fmap, 2, 'avg'))
print('global avg pool:', fmap.mean())

**What to notice:** **max** pooling preserves the sharp peaks (good for detecting whether a
feature is *present*), while **average** pooling smooths and retains overall intensity (good for
summarizing). Max is the classic default; **global average pooling** (averaging a whole feature map to
one number) is the modern replacement for fully-connected heads.

## Gotchas & tradeoffs

- **Pooling discards spatial precision.** Great for classification (*what* is in the image), bad for
  segmentation (*where*) — which is why U-Nets add skip connections to recover location.
- **Max vs average.** Max keeps the strongest response (feature presence); average keeps the mean
  (feature intensity). Pick to match the task.
- **Pooling is being replaced.** Many modern nets use **strided convolutions** to downsample (learnable)
  instead of fixed pooling, and **global average pooling** instead of big dense heads.
- **Receptive field must cover the object.** If a net's receptive field is smaller than the objects it
  must recognize, it literally can't see them — a design constraint, not a detail.

In [ ]:
# Max vs average pooling on the same window: different summaries
window = np.array([[1.0, 3.0], [2.0, 9.0]])
print('window:', window.tolist())
print('max  pool ->', window.max(), '(keeps the strong response 9)')
print('avg  pool ->', window.mean(), '(smooths to the mean)')

# Two 3x3 convs vs one 5x5: same receptive field, fewer params, more non-linearity
print(f'\ntwo 3x3 convs: {2*3*3} weights, receptive field 5x5, 2 ReLUs')
print(f'one 5x5 conv : {5*5} weights, receptive field 5x5, 1 ReLU')

**What to notice:** on the same window, max returns 9 (the peak) while average returns 3.75 (the
mean) — different summaries for different goals. And two 3×3 convs (18 weights, 2 non-linearities)
beat one 5×5 conv (25 weights, 1 non-linearity) at the same receptive field — the depth-over-width
principle that shaped VGG and everything after.

## Key takeaways

- **Pooling** shrinks spatial size, adds translation invariance, and cuts compute.
- Architectures evolved: LeNet → AlexNet → VGG (deep 3×3) → ResNet (**skip connections**).
- **Skip connections** let gradients bypass layers, enabling 100+ layer networks.
- Global average pooling replaces giant fully-connected heads in modern CNNs.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — 2×2 max pooling

Max pooling keeps the strongest activation in each block — shrinking the map 2× per side and buying a little translation tolerance. Implement the standard 2×2, stride-2 version. The last check demonstrates the tolerance: moving the strongest value *within its block* leaves the pooled output unchanged.

In [ ]:
def max_pool(img):
    """2x2 max pooling with stride 2 (assume even dimensions)."""
    img = np.asarray(img, dtype=float)
    H, W = img.shape
    out = np.zeros((H // 2, W // 2))

    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            # TODO(you): max of the 2x2 block at rows 2i:2i+2, cols 2j:2j+2
            out[i, j] = ...

    return out

In [ ]:
# Checks — run me
img = np.array([
    [1, 3, 2, 0],
    [4, 2, 1, 1],
    [0, 1, 8, 2],
    [2, 1, 0, 3],
])
assert max_pool(img).shape == (2, 2), "4x4 -> 2x2"
assert np.allclose(max_pool(img), [[4, 2], [2, 8]]), "max of each 2x2 block"

shifted = img.copy(); shifted[0, 0], shifted[0, 1] = 3, 1   # move values within the top-left block
assert np.allclose(max_pool(shifted), max_pool(img)), \
    "moving the max within its own block doesn't change the pooled output (translation tolerance)"

# Edge cases (DML tests.json style)
tiny = np.array([[1.0, 5.0], [3.0, 2.0]])   # exactly one 2x2 window
assert max_pool(tiny).shape == (1, 1), "smallest possible input: one window -> 1x1 output"
assert max_pool(tiny)[0, 0] == 5.0, "picks the single max in the only window"

tie = np.array([[7.0, 7.0], [7.0, 7.0]])    # every value tied for the max
assert max_pool(tie)[0, 0] == 7.0, "a four-way tie still returns that (shared) max value"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def max_pool(img):
    img = np.asarray(img, dtype=float)
    H, W = img.shape
    out = np.zeros((H // 2, W // 2))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.max(img[2 * i:2 * i + 2, 2 * j:2 * j + 2])
    return out
```

</details>

### Exercise 2 — Receptive-field growth

How much of the input does one output pixel *see*? Walk the layers forward, tracking the receptive field and the cumulative stride ("jump"):

$$rf \mathrel{+}= (k - 1) \cdot jump, \qquad jump \mathrel{*}= s$$

starting from $rf = jump = 1$. The checks verify VGG's famous economy: **three stacked 3×3 convs see exactly as far as one 7×7** — with fewer parameters and two extra nonlinearities — and that a stride-2 pool doubles the growth rate of everything after it.

In [ ]:
def receptive_field(layers):
    """Receptive field of the final output, given layers as (kernel, stride) pairs."""
    rf, jump = 1, 1
    for k, s in layers:
        # TODO(you): grow the receptive field by (k - 1) * jump
        rf = ...

        # TODO(you): multiply the jump by this layer's stride
        jump = ...

    return rf

In [ ]:
# Checks — run me
assert receptive_field([(3, 1)]) == 3, "one 3x3 conv sees 3x3"
assert receptive_field([(3, 1), (3, 1)]) == 5, "two stacked 3x3 convs see 5x5"
assert receptive_field([(3, 1), (3, 1), (3, 1)]) == 7, "three see 7x7 — same as one 7x7, fewer params"
assert receptive_field([(7, 1)]) == receptive_field([(3, 1), (3, 1), (3, 1)]), "VGG's trick"
assert receptive_field([(3, 1), (2, 2), (3, 1)]) == 8, "a stride-2 pool doubles later layers' growth"

# Edge cases
assert receptive_field([]) == 1, "no layers: a pixel only sees itself"
assert receptive_field([(1, 1)]) == 1, "a 1x1 kernel adds nothing to the receptive field"
assert receptive_field([(1, 1), (1, 1), (1, 1)]) == 1, "stacking 1x1 convs still sees just 1 pixel"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def receptive_field(layers):
    rf, jump = 1, 1
    for k, s in layers:
        rf = rf + (k - 1) * jump
        jump = jump * s
    return rf
```

</details>

### Extra practice — DML #114: global average pooling

This lesson's takeaways mention that **global average pooling** collapses each
channel of a feature map to a single number, replacing the giant fully-connected
heads used in older architectures (see "Key takeaways" above). Implement it for
an `(H, W, C)` feature-map tensor: the result is a `(C,)` vector, one number per
channel — the mean over the entire spatial extent of that channel.

In [ ]:
def global_avg_pool(x: np.ndarray) -> np.ndarray:
    """DML #114 -- Global Average Pooling.
    x: (H, W, C) feature maps. Returns: (C,) array, the spatial mean per channel.
    """
    # TODO(you): average over the height and width axes, keep the channel axis
    return ...

In [ ]:
# Checks — run me (DML's own test cases + edge cases)
x = np.array([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]])
assert np.allclose(global_avg_pool(x), [5.5, 6.5, 7.5]), "mean per channel over all 4 spatial positions"

# Edge case: single spatial position (1x1 feature map) -> pooling is a no-op per channel
assert np.allclose(global_avg_pool(np.array([[[100, 200]]])), [100., 200.]), \
    "1x1 spatial size: GAP just returns that pixel's values"

# Edge case: single channel
assert np.allclose(global_avg_pool(np.ones((3, 3, 1))), [1.]), "single-channel input -> shape (1,)"
assert global_avg_pool(np.ones((3, 3, 1))).shape == (1,)

# Edge case: many channels, larger spatial extent
rng = np.random.default_rng(0)
big = rng.standard_normal((5, 5, 8))
pooled = global_avg_pool(big)
assert pooled.shape == (8,), "one number per channel, however large the spatial extent"
assert np.allclose(pooled, big.reshape(-1, 8).mean(axis=0)), "matches a flatten-then-mean-per-column"
print("✅ DML #114 passed")

<details>
<summary>💡 Show solution</summary>

```python
def global_avg_pool(x: np.ndarray) -> np.ndarray:
    return np.mean(x, axis=(0, 1))
```

</details>

### Extra practice — DML #130: a toy CNN, trained end-to-end with backprop

Everything above (convolution, pooling, receptive fields, GAP) has been about
the *forward* pass. This closing exercise, inspired by [Deep-ML #130](https://github.com/Open-Deep-ML/DML-OpenProblem),
asks you to derive and implement the **backward** pass too, and actually train
the tiny network with gradient descent.

The architecture is kept deliberately small — one conv layer, one 2×2 max-pool,
one dense layer — not a deep network:

$$
X \xrightarrow{\text{conv } 3\times3} Z_{conv} \xrightarrow{\text{ReLU}} A_{conv}
\xrightarrow{\text{max-pool } 2\times2} A_{pool} \xrightarrow{\text{flatten + dense}} Z_{dense}
\xrightarrow{\text{softmax}} \hat{y}
$$

with cross-entropy loss. The conv and max-pool forward helpers are provided (you
built equivalent versions of both earlier in this course) — your job is the
**backward pass**: back-prop through the dense layer, un-pool the gradient
through the max-pool's argmax positions, mask by the ReLU, and accumulate the
conv-kernel gradient the same way a conv forward pass accumulates its output
(swap the roles of the input and the upstream gradient).

**Check:** after enough epochs of full-batch gradient descent, the cross-entropy
loss on this toy dataset should fall by more than half.

In [ ]:
def relu(z):
    return np.maximum(z, 0)

def softmax(z):
    e = np.exp(z - np.max(z))
    return e / np.sum(e)

def conv2d_valid(x, k):
    """Valid (no padding, stride 1) 2D convolution -- same as earlier in this course."""
    kh, kw = k.shape
    H, W = x.shape
    oh, ow = H - kh + 1, W - kw + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = np.sum(x[i:i + kh, j:j + kw] * k)
    return out

def maxpool2x2(x):
    """2x2, stride-2 max pool. Also returns the argmax position within each
    window, needed to route the gradient during the backward pass."""
    H, W = x.shape
    oh, ow = H // 2, W // 2
    out = np.zeros((oh, ow))
    argmax = np.zeros((oh, ow, 2), dtype=int)
    for i in range(oh):
        for j in range(ow):
            window = x[2 * i:2 * i + 2, 2 * j:2 * j + 2]
            idx = np.unravel_index(np.argmax(window), window.shape)
            argmax[i, j] = idx
            out[i, j] = window[idx]
    return out, argmax

def maxpool2x2_backward(dpool, argmax, shape):
    """Route each pooled-output gradient back to the single input position
    that was the max in its window; everywhere else gets zero gradient."""
    dx = np.zeros(shape)
    oh, ow = dpool.shape
    for i in range(oh):
        for j in range(ow):
            di, dj = argmax[i, j]
            # TODO(you): accumulate dpool[i, j] at the argmax position (2i+di, 2j+dj)
            ...
    return dx

def train_toy_cnn(X, y, epochs, lr, kernel_size=3, seed=0):
    """Train conv(kernel_size, 1 filter) -> ReLU -> maxpool(2x2) -> dense -> softmax
    with cross-entropy loss, via full-batch gradient descent.

    X: (n_samples, H, W) grayscale images. y: (n_samples, num_classes) one-hot labels.
    Returns the trained parameters and the per-epoch mean loss history.
    """
    rng = np.random.default_rng(seed)
    n, H, W = X.shape
    num_classes = y.shape[1]
    conv_out_h, conv_out_w = H - kernel_size + 1, W - kernel_size + 1
    pool_h, pool_w = conv_out_h // 2, conv_out_w // 2
    flat_size = pool_h * pool_w

    W_conv = rng.standard_normal((kernel_size, kernel_size)) * 0.1
    b_conv = 0.0
    W_dense = rng.standard_normal((flat_size, num_classes)) * 0.1
    b_dense = np.zeros(num_classes)

    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for i in range(n):
            x = X[i]
            # --- forward ---
            z_conv = conv2d_valid(x, W_conv) + b_conv
            a_conv = relu(z_conv)
            a_pool, argmax = maxpool2x2(a_conv)
            a_flat = a_pool.flatten()
            z_dense = a_flat @ W_dense + b_dense
            probs = softmax(z_dense)
            epoch_loss += -np.sum(y[i] * np.log(probs + 1e-12))

            # --- backward ---
            # TODO(you): gradient of cross-entropy + softmax w.r.t. z_dense
            # (the classic simplification: predicted probabilities minus the one-hot label)
            dz_dense = ...

            dW_dense = np.outer(a_flat, dz_dense)
            db_dense = dz_dense

            # TODO(you): gradient flowing back into the flattened pooled activations
            da_flat = ...
            da_pool = da_flat.reshape(pool_h, pool_w)

            da_conv = maxpool2x2_backward(da_pool, argmax, a_conv.shape)

            # TODO(you): backprop through ReLU (zero out gradient where z_conv <= 0)
            dz_conv = ...

            dW_conv = np.zeros_like(W_conv)
            for ii in range(kernel_size):
                for jj in range(kernel_size):
                    dW_conv[ii, jj] = np.sum(dz_conv * x[ii:ii + conv_out_h, jj:jj + conv_out_w])
            db_conv = np.sum(dz_conv)

            # --- SGD update ---
            W_conv -= lr * dW_conv
            b_conv -= lr * db_conv
            W_dense -= lr * dW_dense
            b_dense -= lr * db_dense
        losses.append(epoch_loss / n)

    return W_conv, b_conv, W_dense, b_dense, losses

In [ ]:
# Checks — run me
rng = np.random.default_rng(1)
X_toy = rng.standard_normal((4, 6, 6))
y_toy = np.zeros((4, 2))
y_toy[:2, 0] = 1   # first two examples: class 0
y_toy[2:, 1] = 1   # last two examples: class 1

W_conv, b_conv, W_dense, b_dense, losses = train_toy_cnn(X_toy, y_toy, epochs=60, lr=0.2, kernel_size=3, seed=0)

assert W_conv.shape == (3, 3), "one 3x3 filter"
assert W_dense.shape == (4, 2), "6x6 img -> 4x4 conv -> 2x2 pool -> 4 flattened features, 2 classes"
assert len(losses) == 60
assert losses[-1] < losses[0], "cross-entropy loss should go down as training proceeds"
assert losses[-1] < 0.5 * losses[0], "with 60 full-batch epochs on 4 easy examples, loss should more than halve"

# Edge case: a single training example still trains without shape errors
W_conv1, b_conv1, W_dense1, b_dense1, losses1 = train_toy_cnn(X_toy[:1], y_toy[:1], epochs=10, lr=0.1, kernel_size=3, seed=0)
assert losses1[-1] <= losses1[0], "loss shouldn't increase on a single example either"
print(f"✅ DML #130 passed (loss {losses[0]:.4f} -> {losses[-1]:.4f} over {len(losses)} epochs)")

<details>
<summary>💡 Show solution</summary>

```python
def maxpool2x2_backward(dpool, argmax, shape):
    dx = np.zeros(shape)
    oh, ow = dpool.shape
    for i in range(oh):
        for j in range(ow):
            di, dj = argmax[i, j]
            dx[2 * i + di, 2 * j + dj] += dpool[i, j]
    return dx

def train_toy_cnn(X, y, epochs, lr, kernel_size=3, seed=0):
    rng = np.random.default_rng(seed)
    n, H, W = X.shape
    num_classes = y.shape[1]
    conv_out_h, conv_out_w = H - kernel_size + 1, W - kernel_size + 1
    pool_h, pool_w = conv_out_h // 2, conv_out_w // 2
    flat_size = pool_h * pool_w

    W_conv = rng.standard_normal((kernel_size, kernel_size)) * 0.1
    b_conv = 0.0
    W_dense = rng.standard_normal((flat_size, num_classes)) * 0.1
    b_dense = np.zeros(num_classes)

    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for i in range(n):
            x = X[i]
            z_conv = conv2d_valid(x, W_conv) + b_conv
            a_conv = relu(z_conv)
            a_pool, argmax = maxpool2x2(a_conv)
            a_flat = a_pool.flatten()
            z_dense = a_flat @ W_dense + b_dense
            probs = softmax(z_dense)
            epoch_loss += -np.sum(y[i] * np.log(probs + 1e-12))

            dz_dense = probs - y[i]                 # softmax + cross-entropy gradient
            dW_dense = np.outer(a_flat, dz_dense)
            db_dense = dz_dense
            da_flat = dz_dense @ W_dense.T           # gradient into the flattened pooled activations
            da_pool = da_flat.reshape(pool_h, pool_w)
            da_conv = maxpool2x2_backward(da_pool, argmax, a_conv.shape)
            dz_conv = da_conv * (z_conv > 0)         # ReLU backward

            dW_conv = np.zeros_like(W_conv)
            for ii in range(kernel_size):
                for jj in range(kernel_size):
                    dW_conv[ii, jj] = np.sum(dz_conv * x[ii:ii + conv_out_h, jj:jj + conv_out_w])
            db_conv = np.sum(dz_conv)

            W_conv -= lr * dW_conv
            b_conv -= lr * db_conv
            W_dense -= lr * dW_dense
            b_dense -= lr * db_dense
        losses.append(epoch_loss / n)

    return W_conv, b_conv, W_dense, b_dense, losses
```

</details>